# 23 — Stacking: meta-modelo sobre 5 modelos base
## PRT Seguros

Treinamos 5 modelos de base (CatBoost, XGBoost, LightGBM, Random Forest, Regressão Logística),
cada um com bagging 5-fold (script `stacking_script.py`, mesma lógica dos notebooks `19`/`20`).
Agora treinamos um **meta-modelo** (Regressão Logística) que aprende o peso ótimo de cada um deles
a partir das previsões *out-of-fold* — em vez de uma média fixa (blending), o stacking deixa o
próprio dado decidir o quanto confiar em cada modelo.

AUC-ROC out-of-fold de cada modelo de base:

| Modelo | AUC-ROC OOF |
|---|---|
| CatBoost | 0.8259 |
| XGBoost | 0.8249 |
| LightGBM | 0.8224 |
| Random Forest | 0.8226 |
| Regressão Logística | 0.8124 |


## 1. Carregar as previsões OOF e as previsões no Kaggle

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

oof = pd.read_csv("dados_processados/stacking_oof.csv")
kaggle_preds = pd.read_csv("dados_processados/stacking_kaggle_preds.csv")

MODELOS_BASE = ["catboost", "xgboost", "lightgbm", "random_forest", "logistic_regression"]
X_meta = oof[MODELOS_BASE]
y_meta = oof[TARGET_COL]

print(f"Matriz de meta-features: {X_meta.shape}")
X_meta.head()


Matriz de meta-features: (100000, 5)


,catboost,xgboost,lightgbm,random_forest,logistic_regression
0,0.207784,0.217743,0.139272,0.151456,0.281748
1,0.102289,0.102124,0.117997,0.085444,0.095121
2,0.555855,0.537105,0.227703,0.424030,0.516534
3,0.139907,0.128762,0.129327,0.101317,0.182976
4,0.123734,0.132896,0.125301,0.092004,0.159784


## 2. Treinar o meta-modelo (Regressão Logística) com CV honesto

Como as previsões de entrada já são *out-of-fold*, ainda fazemos uma CV na própria etapa do
meta-modelo (5-fold) pra ter uma estimativa de AUC sem viés — o meta-modelo nunca vê, na hora de
prever, uma linha que usou para se ajustar.

In [2]:
meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
meta = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

meta_oof_proba = cross_val_predict(meta, X_meta, y_meta, cv=meta_cv, method="predict_proba")[:, 1]
auc_stacking = roc_auc_score(y_meta, meta_oof_proba)

print(f"AUC-ROC do STACKING (meta-modelo, CV honesta): {auc_stacking:.4f}")
print(f"Referência — melhor modelo individual (CatBoost): 0.8259")
print(f"Referência — melhor blend simples (bagging one-hot): 0.7383 no Kaggle público")


AUC-ROC do STACKING (meta-modelo, CV honesta): 0.8256
Referência — melhor modelo individual (CatBoost): 0.8259
Referência — melhor blend simples (bagging one-hot): 0.7383 no Kaggle público


## 3. Treinar o meta-modelo final (em 100% do OOF) e ver os pesos aprendidos

In [3]:
meta_final = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta_final.fit(X_meta, y_meta)

pesos = pd.Series(meta_final.coef_[0], index=MODELOS_BASE).sort_values(ascending=False)
print("Pesos (coeficientes) aprendidos pelo meta-modelo:")
print(pesos)
print(f"\nIntercepto: {meta_final.intercept_[0]:.4f}")


Pesos (coeficientes) aprendidos pelo meta-modelo:
catboost               2.859990
xgboost                1.540047
random_forest          0.731028
logistic_regression    0.085635
lightgbm               0.005979
dtype: float64

Intercepto: -4.5236


## 4. Aplicar o meta-modelo nas previsões do Kaggle e gerar a submissão

In [4]:
X_kaggle_meta = kaggle_preds[MODELOS_BASE]
proba_final = meta_final.predict_proba(X_kaggle_meta)[:, 1]

submissao = pd.DataFrame({
    "Id": kaggle_preds[ID_COL],
    "probabilidade_churn": proba_final,
})
submissao.to_csv("submissions/submission_stacking.csv", index=False)
print(f"AUC-ROC stacking (CV): {auc_stacking:.4f}")
print("Salvo: submissions/submission_stacking.csv")
submissao.head()


AUC-ROC stacking (CV): 0.8256
Salvo: submissions/submission_stacking.csv


,Id,probabilidade_churn
0,221300000002,0.018543
1,221300000020,0.036595
2,221300000097,0.024142
3,221300000148,0.078589
4,221300000166,0.053469
